In [204]:
 # importação do pandas e numpy

import pandas as pd
import numpy as np

In [205]:
# Criação de dataframe com o conteúdo bruto do arquivo csv

caminhoArquivo ="data/RelatorioTecnologia.csv"
df_csv_bruto = pd.read_csv(caminhoArquivo, encoding="cp1252")

df_csv_bruto = df_csv_bruto.rename(columns={'DIA TRABALHADO:': 'DATA'})
df_csv_bruto['DATA'] = pd.to_datetime(df_csv_bruto['DATA'], format='%d/%m/%Y')
display(df_csv_bruto)

,ID,DATA,MOCHILINKS,RETORNO VIVOS,MULTIVIEW3,MULTIVIEW2,GLOBOPLAY,MULTIVIEW,CLEARCOM,RESUMO DA OCORRÊNCIA
0,1,2026-01-01,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN
1,2,2026-01-02,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN
2,3,2026-01-03,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN
3,4,2026-01-04,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN
4,5,2026-01-05,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN
...,...,...,...,...,...,...,...,...,...,...
409,410,2026-09-28,CONFORME,NÃO CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,Falha no retorno de áudio externo no LiveU 3 d...
410,411,2026-09-28,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN
411,412,2026-09-28,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN
412,413,2026-09-29,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,CONFORME,NaN


In [206]:
# Criando a dimensão "EQUIPAMENTO"
df_equipamento = pd.DataFrame({
    'id_equipamento': ['EQ_MOCHILINKS', 'EQ_RETORNO_VIVOS', 'EQ_MULTIVIEW3', 'EQ_MULTIVIEW2', 'EQ_GLOBOPLAY', 'EQ_MULTIVIEW', 'EQ_CLEARCOM'],
    'nome_coluna_csv': ['MOCHILINKS', 'RETORNO VIVOS', 'MULTIVIEW3', 'MULTIVIEW2', 'GLOBOPLAY', 'MULTIVIEW', 'CLEARCOM'],
    'nome_ativo': ['Mochilinks LiveU', 'Retorno de Vivos', 'Multiview 3 (CPA)', 'Multiview 2', 'Globoplay', 'Multiview Central', 'Matriz Clear-Com'],
    'categoria_familia': ['Transmissão Externa', 'Retorno e Monitoração', 'Mosaico / Multiview', 'Mosaico / Multiview', 'Distribuição Digital', 'Mosaico / Multiview', 'Intercomunicação de Voz'],
    'limite_toleravel': [3, 4, 2, 2, 2, 2, 2] # Limites para o alerta RF10
})

print(" -- Tabela da dimensão EQUIPAMENTO -- ")
display(df_equipamento)


 -- Tabela da dimensão EQUIPAMENTO -- 


,id_equipamento,nome_coluna_csv,nome_ativo,categoria_familia,limite_toleravel
0,EQ_MOCHILINKS,MOCHILINKS,Mochilinks LiveU,Transmissão Externa,3
1,EQ_RETORNO_VIVOS,RETORNO VIVOS,Retorno de Vivos,Retorno e Monitoração,4
2,EQ_MULTIVIEW3,MULTIVIEW3,Multiview 3 (CPA),Mosaico / Multiview,2
3,EQ_MULTIVIEW2,MULTIVIEW2,Multiview 2,Mosaico / Multiview,2
4,EQ_GLOBOPLAY,GLOBOPLAY,Globoplay,Distribuição Digital,2
5,EQ_MULTIVIEW,MULTIVIEW,Multiview Central,Mosaico / Multiview,2
6,EQ_CLEARCOM,CLEARCOM,Matriz Clear-Com,Intercomunicação de Voz,2


In [207]:
# Criando a dimensão "TEMPO"

# Extração de datas únicas 
df_csv_bruto['DATA'] = pd.to_datetime(df_csv_bruto['DATA'], format='%d/%m/%Y')

datas_unicas = df_csv_bruto['DATA'].drop_duplicates().sort_values().reset_index(drop=True)

df_tempo = pd.DataFrame({
    'data_referencia': datas_unicas,
    'mes': datas_unicas.dt.month,
    'ano': datas_unicas.dt.year,
    # Cria o bloco "Q1_2026", "Q2_2026", etc. (Q = Quarter/Trimestre)
    'trimestre_bloco': 'Q' + datas_unicas.dt.quarter.astype(str) + '_' + datas_unicas.dt.year.astype(str)
})
df_tempo['id_bloco'] = range(1, len(df_tempo) + 1)

print("-- Tabela da dimensão TEMPO -- ")
display(df_tempo)


-- Tabela da dimensão TEMPO -- 


,data_referencia,mes,ano,trimestre_bloco,id_bloco
0,2026-01-01,1,2026,Q1_2026,1
1,2026-01-02,1,2026,Q1_2026,2
2,2026-01-03,1,2026,Q1_2026,3
3,2026-01-04,1,2026,Q1_2026,4
4,2026-01-05,1,2026,Q1_2026,5
...,...,...,...,...,...
229,2026-09-26,9,2026,Q3_2026,230
230,2026-09-27,9,2026,Q3_2026,231
231,2026-09-28,9,2026,Q3_2026,232
232,2026-09-29,9,2026,Q3_2026,233


In [208]:
# Criação da tabela FATO

df_long = df_csv_bruto.melt(
    id_vars=['ID', 'DATA', 'RESUMO DA OCORRÊNCIA'], 
    value_vars=df_equipamento['nome_coluna_csv'].tolist(),
    var_name='nome_coluna_csv',
    value_name='status_conformidade'
)

# limpeza de valores nulos do csv original
df_long['RESUMO DA OCORRÊNCIA'] = df_long['RESUMO DA OCORRÊNCIA'].fillna('Sem intercorrências')

# Interligando os IDs (Fazendo os joins com as dimensões)

df_ocorrencias = df_long.merge(df_equipamento[['nome_coluna_csv', 'id_equipamento']], on='nome_coluna_csv', how='left')

# Trazendo o id_bloco
df_ocorrencias = df_ocorrencias.merge(df_tempo[['data_referencia', 'id_bloco']], left_on='DATA', right_on='data_referencia', how='left')

# 4.4 Organizando a tabela final e gerando o ID da ocorrência
df_ocorrencias = df_ocorrencias.rename(columns={
    'ID': 'id_inspecao_origem',
    'RESUMO DA OCORRÊNCIA': 'descricao_defeito'
})

# Cria a Chave Primária da tabela fato
df_ocorrencias['id_ocorrencia'] = range(1, len(df_ocorrencias) + 1) 

# Seleciona apenas as colunas do Modelo Lógico Relacional
df_ocorrencias = df_ocorrencias[[
    'id_ocorrencia', 'id_inspecao_origem', 'id_equipamento', 'id_bloco', 
    'data_referencia', 'status_conformidade', 'descricao_defeito'
]]


print("-- TABELA DE OCORRÊNCIAS --")
display(df_ocorrencias)

-- TABELA DE OCORRÊNCIAS --


,id_ocorrencia,id_inspecao_origem,id_equipamento,id_bloco,data_referencia,status_conformidade,descricao_defeito
0,1,1,EQ_MOCHILINKS,1,2026-01-01,CONFORME,Sem intercorrências
1,2,2,EQ_MOCHILINKS,2,2026-01-02,CONFORME,Sem intercorrências
2,3,3,EQ_MOCHILINKS,3,2026-01-03,CONFORME,Sem intercorrências
3,4,4,EQ_MOCHILINKS,4,2026-01-04,CONFORME,Sem intercorrências
4,5,5,EQ_MOCHILINKS,5,2026-01-05,CONFORME,Sem intercorrências
...,...,...,...,...,...,...,...
2893,2894,410,EQ_CLEARCOM,232,2026-09-28,CONFORME,Falha no retorno de áudio externo no LiveU 3 d...
2894,2895,411,EQ_CLEARCOM,232,2026-09-28,CONFORME,Sem intercorrências
2895,2896,412,EQ_CLEARCOM,232,2026-09-28,CONFORME,Sem intercorrências
2896,2897,413,EQ_CLEARCOM,233,2026-09-29,CONFORME,Sem intercorrências


In [209]:
# RANKING
print(" ############# Ranking de criticidade #############")
df_falhas = df_ocorrencias[df_ocorrencias['status_conformidade'] == 'NÃO CONFORME']

# Calcula a quantidade de falhas por equipamento
qtd_falhas =df_falhas['id_equipamento'].value_counts()

# Calcula a porcentagem de falhas
taxa_percentual = (df_falhas['id_equipamento'].value_counts(normalize=True) * 100).round(1)

print("-- Quantidade de Falhas por Equipamento --")
display(qtd_falhas)

print(" -- Taxa de falhas -- ")
display(taxa_percentual)



 ############# Ranking de criticidade #############
-- Quantidade de Falhas por Equipamento --


id_equipamento
EQ_RETORNO_VIVOS    10
EQ_MOCHILINKS        5
EQ_MULTIVIEW3        3
EQ_MULTIVIEW         3
EQ_CLEARCOM          3
EQ_GLOBOPLAY         2
Name: count, dtype: int64

 -- Taxa de falhas -- 


id_equipamento
EQ_RETORNO_VIVOS    38.5
EQ_MOCHILINKS       19.2
EQ_MULTIVIEW3       11.5
EQ_MULTIVIEW        11.5
EQ_CLEARCOM         11.5
EQ_GLOBOPLAY         7.7
Name: proportion, dtype: float64

In [210]:
# Relatório final de falhas

df_relatorio_falhas = pd.DataFrame({
    'Quantidade de Falhas': qtd_falhas,
    'Taxa Percentual (%)': taxa_percentual
}).reset_index()

# Renomeando a coluna de índice
df_relatorio_falhas = df_relatorio_falhas.rename(columns={'id_equipamento': 'Equipamento'})

# Exibir o resultado


display(df_equipamento)
display(df_ocorrencias)
display(df_tempo)

print("--- RELATÓRIO DE CRITICIDADE E % DE FALHAS ---")
display(df_relatorio_falhas)

,id_equipamento,nome_coluna_csv,nome_ativo,categoria_familia,limite_toleravel
0,EQ_MOCHILINKS,MOCHILINKS,Mochilinks LiveU,Transmissão Externa,3
1,EQ_RETORNO_VIVOS,RETORNO VIVOS,Retorno de Vivos,Retorno e Monitoração,4
2,EQ_MULTIVIEW3,MULTIVIEW3,Multiview 3 (CPA),Mosaico / Multiview,2
3,EQ_MULTIVIEW2,MULTIVIEW2,Multiview 2,Mosaico / Multiview,2
4,EQ_GLOBOPLAY,GLOBOPLAY,Globoplay,Distribuição Digital,2
5,EQ_MULTIVIEW,MULTIVIEW,Multiview Central,Mosaico / Multiview,2
6,EQ_CLEARCOM,CLEARCOM,Matriz Clear-Com,Intercomunicação de Voz,2


,id_ocorrencia,id_inspecao_origem,id_equipamento,id_bloco,data_referencia,status_conformidade,descricao_defeito
0,1,1,EQ_MOCHILINKS,1,2026-01-01,CONFORME,Sem intercorrências
1,2,2,EQ_MOCHILINKS,2,2026-01-02,CONFORME,Sem intercorrências
2,3,3,EQ_MOCHILINKS,3,2026-01-03,CONFORME,Sem intercorrências
3,4,4,EQ_MOCHILINKS,4,2026-01-04,CONFORME,Sem intercorrências
4,5,5,EQ_MOCHILINKS,5,2026-01-05,CONFORME,Sem intercorrências
...,...,...,...,...,...,...,...
2893,2894,410,EQ_CLEARCOM,232,2026-09-28,CONFORME,Falha no retorno de áudio externo no LiveU 3 d...
2894,2895,411,EQ_CLEARCOM,232,2026-09-28,CONFORME,Sem intercorrências
2895,2896,412,EQ_CLEARCOM,232,2026-09-28,CONFORME,Sem intercorrências
2896,2897,413,EQ_CLEARCOM,233,2026-09-29,CONFORME,Sem intercorrências


,data_referencia,mes,ano,trimestre_bloco,id_bloco
0,2026-01-01,1,2026,Q1_2026,1
1,2026-01-02,1,2026,Q1_2026,2
2,2026-01-03,1,2026,Q1_2026,3
3,2026-01-04,1,2026,Q1_2026,4
4,2026-01-05,1,2026,Q1_2026,5
...,...,...,...,...,...
229,2026-09-26,9,2026,Q3_2026,230
230,2026-09-27,9,2026,Q3_2026,231
231,2026-09-28,9,2026,Q3_2026,232
232,2026-09-29,9,2026,Q3_2026,233


--- RELATÓRIO DE CRITICIDADE E % DE FALHAS ---


,Equipamento,Quantidade de Falhas,Taxa Percentual (%)
0,EQ_RETORNO_VIVOS,10,38.5
1,EQ_MOCHILINKS,5,19.2
2,EQ_MULTIVIEW3,3,11.5
3,EQ_MULTIVIEW,3,11.5
4,EQ_CLEARCOM,3,11.5
5,EQ_GLOBOPLAY,2,7.7
